In [88]:
from transformers import pipeline
from nltk import sent_tokenize 
import nltk
import torch
from glob import glob 
import pandas as pd
import numpy as np

In [3]:
nltk.download("punkt")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\shawa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [4]:
model_name = "facebook/bart-large-mnli"
device = 0 if torch.cuda.is_available() else 'cpu'


In [5]:
device

'cpu'

In [6]:
def load_model(device):
    theme_classifier = pipeline(
        "zero-shot-classification",
        model=model_name,
        device=device
    )

    return theme_classifier

In [7]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

theme_classifier = load_model(device)

c:\Users\shawa\Downloads\football-analysis\venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [8]:
theme_list = ["friendship","hope","sacrifice","battle","self development","betrayal","love","dialogue"]

In [9]:
theme_classifier(
    "I gave him a right hook then a left jab",
    theme_list,
    multi_label=True
)

{'sequence': 'I gave him a right hook then a left jab',
 'labels': ['battle',
  'self development',
  'hope',
  'sacrifice',
  'dialogue',
  'betrayal',
  'love',
  'friendship'],
 'scores': [0.9121252298355103,
  0.4750025272369385,
  0.08781788498163223,
  0.04500041902065277,
  0.02013299986720085,
  0.012040478177368641,
  0.004292412661015987,
  0.002817223547026515]}

In [21]:
files = glob('../data/Subtitles/*.ass')
len(files)

218

In [29]:
with open(files[0],'r') as file:
    lines = file.readlines()
    lines = lines[27:]
    lines = [ ",".join(line.split(",")[9:]) for line in lines]
    print(lines)

['A long time ago, a powerful demon fox\\Nappeared with nine tails.\n', 'With its powerful tails,\n', 'it could smash mountains\\Nand create tidal waves.\n', 'A band of Ninjas rose to\\Ndefend their village from attack.\n', 'We have to wait until\\Nthe Fourth Hokage gets here!\n', "We can't let it get any closer\\Nto our village!\n", 'One great Ninja was able to\\Nimprison the monster,\n', 'but died in the process.\n', 'This Ninja was known as…\\Nthe Fourth Hokage.\n', 'Naruto!\n', 'Why did you do such a thing?!\n', "You're really gonna get it this time!\n", "I don't care!\n", 'You know your problem?\n', "You can't do the things I do!\n", 'Only I can do this!\n', "I'm better than all of you!\\NBelieve it!\n", "There's a problem, sir!\n", 'Lord Hokage!\n', 'What is it?\n', 'Did that Naruto do something again?\n', 'Yes. He climbed onto\\Nthe Mountainside Images…\n', 'And he vandalized and\\Ngraffitied all over them!\n', 'Wait!\n', 'Ha ha…\n', 'Why should I?\n', 'Hey, Naruto!\n', 'How did

In [32]:
lines = [line.replace('\\N', ' ') for line in lines]
lines

['A long time ago, a powerful demon fox appeared with nine tails.\n',
 'With its powerful tails,\n',
 'it could smash mountains and create tidal waves.\n',
 'A band of Ninjas rose to defend their village from attack.\n',
 'We have to wait until the Fourth Hokage gets here!\n',
 "We can't let it get any closer to our village!\n",
 'One great Ninja was able to imprison the monster,\n',
 'but died in the process.\n',
 'This Ninja was known as… the Fourth Hokage.\n',
 'Naruto!\n',
 'Why did you do such a thing?!\n',
 "You're really gonna get it this time!\n",
 "I don't care!\n",
 'You know your problem?\n',
 "You can't do the things I do!\n",
 'Only I can do this!\n',
 "I'm better than all of you! Believe it!\n",
 "There's a problem, sir!\n",
 'Lord Hokage!\n',
 'What is it?\n',
 'Did that Naruto do something again?\n',
 'Yes. He climbed onto the Mountainside Images…\n',
 'And he vandalized and graffitied all over them!\n',
 'Wait!\n',
 'Ha ha…\n',
 'Why should I?\n',
 'Hey, Naruto!\n',
 '

In [33]:
" ".join(lines[:10])

"A long time ago, a powerful demon fox appeared with nine tails.\n With its powerful tails,\n it could smash mountains and create tidal waves.\n A band of Ninjas rose to defend their village from attack.\n We have to wait until the Fourth Hokage gets here!\n We can't let it get any closer to our village!\n One great Ninja was able to imprison the monster,\n but died in the process.\n This Ninja was known as… the Fourth Hokage.\n Naruto!\n"

In [39]:
int(files[0].split(" ")[-1].split('.')[0].strip())

1

In [69]:
def load_dataset(data_path):
    files = glob(data_path+"/*.ass")
    scripts = []
    episodes = []
    for path in files:
        with open(path,'r') as file:
            lines = file.readlines()
            lines = lines[27:]
            lines = [ ",".join(line.split(",")[9:]) for line in lines]
        lines = [line.replace('\\N', ' ') for line in lines]
        script = " ".join(lines)
        episode = int(path.split('-')[-1].split('.')[0].strip())
        scripts.append(script)
        episodes.append(episode)
    df = pd.DataFrame({"epsiode_num":episodes, "script":scripts})
    return df
        

In [70]:
df = load_dataset("../data/Subtitles")

In [71]:
print(df)

     epsiode_num                                             script
0              1  A long time ago, a powerful demon fox appeared...
1              2  C'mon!\n Running like a fugitive,\n Being chas...
2              3  C'mon!\n Running like a fugitive,\n Being chas...
3              4  C'mon!\n Running like a fugitive,\n Being chas...
4              5  C'mon!\n Running like a fugitive,\n Being chas...
..           ...                                                ...
213          216  Summoning Jutsu!\n Oh, long time no see.\n We ...
214          217  Gaara.\n Kankuro.\n Gaara.\n Primary Lotus!\n ...
215          218  To think the Leaf’s reinforcements will be you...
216          219  Fly into the wavy and twisted sky, into your h...
217          220  So this is the Shukaku?\n It’s the first time ...

[218 rows x 2 columns]


In [77]:
script = df.iloc[0]['script']
print(script)

A long time ago, a powerful demon fox appeared with nine tails.
 With its powerful tails,
 it could smash mountains and create tidal waves.
 A band of Ninjas rose to defend their village from attack.
 We have to wait until the Fourth Hokage gets here!
 We can't let it get any closer to our village!
 One great Ninja was able to imprison the monster,
 but died in the process.
 This Ninja was known as… the Fourth Hokage.
 Naruto!
 Why did you do such a thing?!
 You're really gonna get it this time!
 I don't care!
 You know your problem?
 You can't do the things I do!
 Only I can do this!
 I'm better than all of you! Believe it!
 There's a problem, sir!
 Lord Hokage!
 What is it?
 Did that Naruto do something again?
 Yes. He climbed onto the Mountainside Images…
 And he vandalized and graffitied all over them!
 Wait!
 Ha ha…
 Why should I?
 Hey, Naruto!
 How did you suddenly get here, lruka Sensei?
 The question is what are you doing here when you should be in class now?
 Now listen, Narut

In [79]:
script_sentences = sent_tokenize(script)
script_sentences[:3]
print(len(script_sentences))

299


In [ ]:
sentence_batch_size=20
script_batches = []
for index in range(0,len(script_sentences),sentence_batch_size):
    sent = " ".join(script_sentences[index:index+sentence_batch_size])
    script_batches.append(sent)
print(len(script_batches))


15


In [83]:
script_batches[:2]

["A long time ago, a powerful demon fox appeared with nine tails. With its powerful tails,\n it could smash mountains and create tidal waves. A band of Ninjas rose to defend their village from attack. We have to wait until the Fourth Hokage gets here! We can't let it get any closer to our village! One great Ninja was able to imprison the monster,\n but died in the process. This Ninja was known as… the Fourth Hokage. Naruto! Why did you do such a thing?! You're really gonna get it this time! I don't care! You know your problem? You can't do the things I do! Only I can do this! I'm better than all of you! Believe it! There's a problem, sir! Lord Hokage! What is it? Did that Naruto do something again?",
 'Yes. He climbed onto the Mountainside Images…\n And he vandalized and graffitied all over them! Wait! Ha ha…\n Why should I? Hey, Naruto! How did you suddenly get here, lruka Sensei? The question is what are you doing here when you should be in class now? Now listen, Naruto. You failed t

In [84]:
theme_output = theme_classifier(
    script_batches[:2],
    theme_list,
    multi_label=True
)

In [85]:
theme_output

[{'sequence': "A long time ago, a powerful demon fox appeared with nine tails. With its powerful tails,\n it could smash mountains and create tidal waves. A band of Ninjas rose to defend their village from attack. We have to wait until the Fourth Hokage gets here! We can't let it get any closer to our village! One great Ninja was able to imprison the monster,\n but died in the process. This Ninja was known as… the Fourth Hokage. Naruto! Why did you do such a thing?! You're really gonna get it this time! I don't care! You know your problem? You can't do the things I do! Only I can do this! I'm better than all of you! Believe it! There's a problem, sir! Lord Hokage! What is it? Did that Naruto do something again?",
  'labels': ['dialogue',
   'betrayal',
   'battle',
   'sacrifice',
   'self development',
   'hope',
   'friendship',
   'love'],
  'scores': [0.9800742268562317,
   0.9396902322769165,
   0.8546884059906006,
   0.7349813580513,
   0.7284974455833435,
   0.19909794628620148,

In [ ]:
themes  = {}

for output in theme_output:
    for label, score in zip(output['labels'] , output["scores"]):
        if label not in themes:
            themes[label] = []
        themes[label].append(score)
themes




{'dialogue': [0.9800742268562317, 0.9370126128196716],
 'betrayal': [0.9396902322769165, 0.645726203918457],
 'battle': [0.8546884059906006, 0.6581322550773621],
 'sacrifice': [0.7349813580513, 0.6258843541145325],
 'self development': [0.7284974455833435, 0.8678207993507385],
 'hope': [0.19909794628620148, 0.20423975586891174],
 'friendship': [0.05922335758805275, 0.08603321015834808],
 'love': [0.04026177525520325, 0.028020622208714485]}

In [92]:
def get_themes_inference(script):
    script_sentences = sent_tokenize(script)

    # Batch Sentence
    sentence_batch_size=20
    script_batches = []
    for index in range(0,len(script_sentences),sentence_batch_size):
        sent = " ".join(script_sentences[index:index+sentence_batch_size])
        script_batches.append(sent)
    
    # Run Model
    theme_output = theme_classifier(
        script_batches[:2],
        theme_list,
        multi_label=True
    )

    # Wrangle Output 
    themes = {}
    for output in theme_output:
        for label,score in zip(output['labels'],output['scores']):
            if label not in themes:
                themes[label] = []
            themes[label].append(score)
    

    themes = {key: np.mean(np.array(value)) for key,value in themes.items()}
    print(themes)
    return themes

In [89]:
df = df.head(2)

In [90]:
df

,epsiode_num,script
0,1,"A long time ago, a powerful demon fox appeared..."
1,2,"C'mon!\n Running like a fugitive,\n Being chas..."


In [93]:
output_themes = df['script'].apply(get_themes_inference)

{'dialogue': np.float64(0.9585434198379517), 'betrayal': np.float64(0.7927082180976868), 'battle': np.float64(0.7564103305339813), 'sacrifice': np.float64(0.6804328560829163), 'self development': np.float64(0.798159122467041), 'hope': np.float64(0.2016688510775566), 'friendship': np.float64(0.07262828387320042), 'love': np.float64(0.034141198731958866)}
{'dialogue': np.float64(0.9606060683727264), 'sacrifice': np.float64(0.5707033574581146), 'self development': np.float64(0.4828104078769684), 'battle': np.float64(0.6848459839820862), 'love': np.float64(0.17326094000600278), 'hope': np.float64(0.1545334211550653), 'friendship': np.float64(0.04626084305346012), 'betrayal': np.float64(0.42994374223053455)}


In [96]:
theme_df = pd.DataFrame(output_themes.to_list())

In [98]:
df[theme_df.columns] = theme_df
df

C:\Users\shawa\AppData\Local\Temp\ipykernel_19316\3417989627.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[theme_df.columns] = theme_df


,epsiode_num,script,dialogue,betrayal,battle,sacrifice,self development,hope,friendship,love
0,1,"A long time ago, a powerful demon fox appeared...",0.958543,0.792708,0.756410,0.680433,0.798159,0.201669,0.072628,0.034141
1,2,"C'mon!\n Running like a fugitive,\n Being chas...",0.960606,0.429944,0.684846,0.570703,0.482810,0.154533,0.046261,0.173261
